# Architecture Selection: Multi-Seed Robustness Check (Protocol-Matched Appendix)

This notebook reruns the architecture-selection candidates across three independent seeds, using the same core Black--Scholes setup and training protocol as the original architecture-selection experiment.

Purpose in the report: appendix robustness evidence only. The main report architecture choice does not need to change unless this protocol-matched rerun produces a materially different conclusion.

Key protocol choices matched to the original experiment:
- Black--Scholes setup: `S0=1`, `K=0.9`, `sigma=0.4`, `T=0.5`, `r=0`, `N=125`.
- Inputs: `log(S/K)` and `tau/T` for normalized shared MLPs.
- Learned scalar premium and terminal MSE loss.
- Adam learning rate `1e-3`.
- Batch size `256`, not `4096`.
- `M_train=100000`, `M_val=20000`, `M_test=50000`, `max_epochs=60`, early-stopping patience `8`, reduce-LR patience `4`.

Outputs are written to a local relative folder under `./mfe_deep_hedging_runs/`. This makes the notebook compatible with the reproducibility runner and avoids hidden Google Drive paths.


## 0. Imports

In [ ]:
import os
import json
import time
import random
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy.stats import norm

import tensorflow as tf
import keras
from keras import layers, callbacks, optimizers

## 1. Global switches

In [ ]:
SMOKE_TEST = False          # Set True for a quick end-to-end plumbing check only
USE_GOOGLE_DRIVE = False    # Keep outputs local/relative so the reproducibility runner can collect them
RUN_ORIGINAL_SEED_SANITY_CHECK = False  # Optional; leave False for the submitted appendix run

# Three pre-committed independent seeds. Each seed controls train/val/test path generation
# and model initialisation for that outer run. Do not change after seeing results.
SEEDS = [2026, 2027, 2028]


## 2. Reproducibility helper

In [ ]:
def set_all_seeds(seed: int) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

## 3. Output directory

In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def make_run_directory() -> Path:
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    # IMPORTANT: use a relative local folder so that the master reproducibility runner
    # can collect CSV/JSON/TXT outputs. Do not write directly to Google Drive here.
    root = Path("./mfe_deep_hedging_runs")
    run_dir = root / f"arch_selection_multiseed_protocol_matched_{timestamp}"
    for sub in ["config", "results", "logs"]:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)
    return run_dir


RUN_DIR = make_run_directory()
print(f"Run directory: {RUN_DIR}")


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def append_log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(RUN_DIR / "logs" / "run_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")


## 4. Experiment configuration

In [ ]:
@dataclass
class GlobalConfig:
    # Black-Scholes minimal-task parameters
    r: float = 0.0
    sigma: float = 0.4
    S0: float = 1.0
    K: float = 0.9
    T: float = 0.5
    N: int = 125

    # Data sizes matched to the original architecture-selection run
    M_train: int = 100_000
    M_val: int = 20_000
    M_test: int = 50_000

    # Training defaults matched to the original architecture-selection run
    max_epochs: int = 60
    early_stopping_patience: int = 8
    reduce_lr_patience: int = 4


if SMOKE_TEST:
    CFG = GlobalConfig(
        N=20, M_train=2_000, M_val=1_000, M_test=2_000,
        max_epochs=4, early_stopping_patience=2, reduce_lr_patience=1,
    )
else:
    CFG = GlobalConfig()

CFG.dt = CFG.T / CFG.N
save_json(asdict(CFG), RUN_DIR / "config" / "global_config.json")
save_json({"SMOKE_TEST": SMOKE_TEST, "SEEDS": SEEDS, "USE_GOOGLE_DRIVE": USE_GOOGLE_DRIVE}, RUN_DIR / "config" / "run_switches.json")
print(f"Config: N={CFG.N}, M_train={CFG.M_train}, M_val={CFG.M_val}, M_test={CFG.M_test}, batch size is set per candidate.")


## 5. Black-Scholes formulas and path simulation

In [ ]:
def bs_call_price(S, K, tau, r, sigma):
    S_arr = np.asarray(S, dtype=float)
    tau_arr = np.asarray(tau, dtype=float)
    out = np.maximum(S_arr - K, 0.0)
    tau_safe = np.maximum(tau_arr, 1e-12)
    d1 = (np.log(np.maximum(S_arr, 1e-12) / K) + (r + 0.5 * sigma**2) * tau_safe) / (sigma * np.sqrt(tau_safe))
    d2 = d1 - sigma * np.sqrt(tau_safe)
    price = S_arr * norm.cdf(d1) - K * np.exp(-r * tau_safe) * norm.cdf(d2)
    return np.where(tau_arr > 1e-12, price, out)


def simulate_gbm(M, N, S0, r, sigma, T, seed=None):
    rng = np.random.default_rng(seed)
    dt = T / N
    Z = rng.standard_normal((M, N)).astype(np.float32)
    log_inc = np.float32((r - 0.5 * sigma**2) * dt) + np.float32(sigma * np.sqrt(dt)) * Z
    log_S = np.float32(np.log(S0)) + np.cumsum(log_inc, axis=1)
    S = np.hstack([np.full((M, 1), S0, dtype=np.float32), np.exp(log_S).astype(np.float32)])
    return S


C0_BS = float(bs_call_price(CFG.S0, CFG.K, CFG.T, CFG.r, CFG.sigma))
print(f"Black-Scholes call price: {C0_BS:.8f}")

## 6. Neural network model classes

In [ ]:
class SharedMLPHedger(keras.Model):
    """Weight-sharing Markov MLP: delta_n = f_theta(log(S_n/K), tau_n/T)."""

    def __init__(self, N, K, T, dt, n_units, n_layers, activation,
                 feature_mode, output_mode, delta_scale=2.0,
                 initial_premium=0.0, l2_reg=0.0, dropout=0.0,
                 name="shared_mlp_hedger"):
        super().__init__(name=name)
        self.N = int(N)
        self.K = float(K)
        self.T = float(T)
        self.dt = float(dt)
        self.feature_mode = feature_mode
        self.output_mode = output_mode
        self.delta_scale = float(delta_scale)
        self.dropout = float(dropout)
        self.tau_vals = tf.constant((T - np.arange(N) * dt).astype(np.float32), dtype=tf.float32)
        self.pi = self.add_weight(
            name="premium", shape=(),
            initializer=keras.initializers.Constant(float(initial_premium)),
            trainable=True, dtype=tf.float32,
        )
        reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
        self.hidden_layers = []
        for i in range(n_layers):
            self.hidden_layers.append(layers.Dense(n_units, activation=activation,
                                                   kernel_regularizer=reg, name=f"dense_{i}"))
            if dropout > 0:
                self.hidden_layers.append(layers.Dropout(dropout, name=f"dropout_{i}"))
        self.out_layer = layers.Dense(1, activation="linear", kernel_regularizer=reg, name="delta_raw")

    def _features(self, S_steps):
        tau_2d = tf.ones_like(S_steps) * self.tau_vals[tf.newaxis, :]
        eps = tf.constant(1e-7, dtype=tf.float32)
        if self.feature_mode == "raw":
            return tf.stack([S_steps, tau_2d], axis=-1)
        if self.feature_mode == "normalized":
            log_m = tf.math.log(tf.maximum(S_steps, eps) / self.K)
            tau_scaled = tau_2d / self.T
            return tf.stack([log_m, tau_scaled], axis=-1)
        raise ValueError(f"Unknown feature_mode: {self.feature_mode}")

    def delta_from_features(self, feat_2d, training=False):
        x = feat_2d
        for layer in self.hidden_layers:
            x = layer(x, training=training)
        raw = self.out_layer(x, training=training)
        if self.output_mode == "sigmoid":
            return tf.sigmoid(raw)
        if self.output_mode == "linear":
            return raw
        if self.output_mode == "scaled_tanh":
            return self.delta_scale * tf.tanh(raw)
        raise ValueError(f"Unknown output_mode: {self.output_mode}")

    def compute_deltas(self, S_path, training=False):
        S_steps = S_path[:, :self.N]
        feat_3d = self._features(S_steps)
        feat_2d = tf.reshape(feat_3d, (-1, feat_3d.shape[-1]))
        delta_flat = self.delta_from_features(feat_2d, training=training)
        return tf.reshape(delta_flat, (-1, self.N))

    def call(self, S_path, training=False):
        S_path = tf.cast(S_path, tf.float32)
        deltas = self.compute_deltas(S_path, training=training)
        dS = S_path[:, 1:] - S_path[:, :self.N]
        pnl = tf.reduce_sum(deltas * dS, axis=1, keepdims=True)
        payoff = tf.maximum(S_path[:, self.N:self.N+1] - self.K, 0.0)
        return self.pi + pnl - payoff


class TimeSeparatedMLPHedger(keras.Model):
    """Time-separated MLP: each rebalancing date has its own subnetwork."""

    def __init__(self, N, K, T, dt, n_units, n_layers, activation,
                 feature_mode, output_mode, delta_scale=2.0,
                 initial_premium=0.0, l2_reg=0.0, dropout=0.0,
                 name="time_separated_mlp_hedger"):
        super().__init__(name=name)
        self.N = int(N)
        self.K = float(K)
        self.T = float(T)
        self.dt = float(dt)
        self.feature_mode = feature_mode
        self.output_mode = output_mode
        self.delta_scale = float(delta_scale)
        self.tau_vals = tf.constant((T - np.arange(N) * dt).astype(np.float32), dtype=tf.float32)
        self.pi = self.add_weight(
            name="premium", shape=(),
            initializer=keras.initializers.Constant(float(initial_premium)),
            trainable=True, dtype=tf.float32,
        )
        reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
        self.step_hidden_layers = []
        self.step_out_layers = []
        for n in range(N):
            hidden = []
            for i in range(n_layers):
                hidden.append(layers.Dense(n_units, activation=activation,
                                           kernel_regularizer=reg, name=f"step{n}_dense{i}"))
                if dropout > 0:
                    hidden.append(layers.Dropout(dropout, name=f"step{n}_dropout{i}"))
            self.step_hidden_layers.append(hidden)
            self.step_out_layers.append(layers.Dense(1, activation="linear",
                                                     kernel_regularizer=reg, name=f"step{n}_out"))

    def _features(self, S_steps):
        tau_2d = tf.ones_like(S_steps) * self.tau_vals[tf.newaxis, :]
        eps = tf.constant(1e-7, dtype=tf.float32)
        if self.feature_mode == "raw":
            return tf.stack([S_steps, tau_2d], axis=-1)
        if self.feature_mode == "normalized":
            log_m = tf.math.log(tf.maximum(S_steps, eps) / self.K)
            tau_scaled = tau_2d / self.T
            return tf.stack([log_m, tau_scaled], axis=-1)
        raise ValueError(f"Unknown feature_mode: {self.feature_mode}")

    def _apply_output(self, raw):
        if self.output_mode == "sigmoid":
            return tf.sigmoid(raw)
        if self.output_mode == "linear":
            return raw
        if self.output_mode == "scaled_tanh":
            return self.delta_scale * tf.tanh(raw)
        raise ValueError(f"Unknown output_mode: {self.output_mode}")

    def compute_deltas(self, S_path, training=False):
        S_path = tf.cast(S_path, tf.float32)
        S_steps = S_path[:, :self.N]
        feat_3d = self._features(S_steps)
        outputs = []
        for n in range(self.N):
            x = feat_3d[:, n, :]
            for layer in self.step_hidden_layers[n]:
                x = layer(x, training=training)
            raw = self.step_out_layers[n](x, training=training)
            outputs.append(self._apply_output(raw))
        return tf.concat(outputs, axis=1)

    def call(self, S_path, training=False):
        S_path = tf.cast(S_path, tf.float32)
        deltas = self.compute_deltas(S_path, training=training)
        dS = S_path[:, 1:] - S_path[:, :self.N]
        pnl = tf.reduce_sum(deltas * dS, axis=1, keepdims=True)
        payoff = tf.maximum(S_path[:, self.N:self.N+1] - self.K, 0.0)
        return self.pi + pnl - payoff


def mse_hedge_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_pred))


def make_hedger_model(cfg):
    common = dict(
        N=CFG.N, K=CFG.K, T=CFG.T, dt=CFG.dt,
        n_units=cfg["n_units"], n_layers=cfg["n_layers"],
        activation=cfg["activation"], feature_mode=cfg["feature_mode"],
        output_mode=cfg["output_mode"], delta_scale=cfg["delta_scale"],
        initial_premium=C0_BS, l2_reg=cfg["l2_reg"], dropout=cfg["dropout"],
    )
    if cfg["architecture"] == "time_separated_mlp":
        return TimeSeparatedMLPHedger(**common)
    return SharedMLPHedger(**common)

## 7. Candidate grid — exactly the 12 Table 1 configurations

In [ ]:
def make_candidate(label, architecture, feature_mode, n_units, n_layers,
                   activation, output_mode, learning_rate=1e-3,
                   batch_size=256, dropout=0.0, l2_reg=0.0, delta_scale=2.0):
    return dict(
        label=label, architecture=architecture, feature_mode=feature_mode,
        n_units=n_units, n_layers=n_layers, activation=activation,
        output_mode=output_mode, learning_rate=learning_rate,
        batch_size=batch_size, dropout=dropout, l2_reg=l2_reg,
        delta_scale=delta_scale,
    )


# Candidate subset aligned to the architecture-selection table used in the report.
# Smoke-test uses a 3-candidate subset.
if SMOKE_TEST:
    TABLE1_CANDIDATES = [
        make_candidate("shared_norm_16u_1L_tanh_sigmoid",  "shared_mlp",        "normalized", 16, 1, "tanh",    "sigmoid"),
        make_candidate("shared_norm_64u_2L_tanh_sigmoid",  "shared_mlp",        "normalized", 64, 2, "tanh",    "sigmoid"),
        make_candidate("time_sep_norm_8u_1L_tanh_sigmoid", "time_separated_mlp","normalized",  8, 1, "tanh",    "sigmoid"),
    ]
else:
    TABLE1_CANDIDATES = [
        # Shared normalized — capacity sweep
        make_candidate("shared_norm_16u_1L_tanh_sigmoid",  "shared_mlp",        "normalized", 16, 1, "tanh",    "sigmoid"),
        make_candidate("shared_norm_16u_2L_tanh_sigmoid",  "shared_mlp",        "normalized", 16, 2, "tanh",    "sigmoid"),
        make_candidate("shared_norm_32u_2L_tanh_sigmoid",  "shared_mlp",        "normalized", 32, 2, "tanh",    "sigmoid"),
        make_candidate("shared_norm_32u_3L_tanh_sigmoid",  "shared_mlp",        "normalized", 32, 3, "tanh",    "sigmoid"),
        make_candidate("shared_norm_64u_2L_tanh_sigmoid",  "shared_mlp",        "normalized", 64, 2, "tanh",    "sigmoid"),
        make_candidate("shared_norm_64u_3L_tanh_sigmoid",  "shared_mlp",        "normalized", 64, 3, "tanh",    "sigmoid"),
        # Activation comparison
        make_candidate("shared_norm_64u_2L_relu_sigmoid",  "shared_mlp",        "normalized", 64, 2, "relu",    "sigmoid"),
        make_candidate("shared_norm_64u_2L_softplus_sigmoid","shared_mlp",      "normalized", 64, 2, "softplus","sigmoid"),
        # Raw input comparison
        make_candidate("shared_raw_32u_2L_tanh_sigmoid",   "shared_mlp",        "raw",        32, 2, "tanh",    "sigmoid"),
        make_candidate("shared_raw_64u_2L_tanh_sigmoid",   "shared_mlp",        "raw",        64, 2, "tanh",    "sigmoid"),
        # Time-separated
        make_candidate("time_sep_norm_8u_1L_tanh_sigmoid", "time_separated_mlp","normalized",  8, 1, "tanh",    "sigmoid"),
        make_candidate("time_sep_norm_16u_1L_tanh_sigmoid","time_separated_mlp","normalized", 16, 1, "tanh",    "sigmoid"),
    ]

print(f"Number of Table 1 candidates: {len(TABLE1_CANDIDATES)}")

# Persist candidate grid for reproducibility.
pd.DataFrame(TABLE1_CANDIDATES).to_csv(RUN_DIR / "config" / "candidate_grid.csv", index=False)
save_json(TABLE1_CANDIDATES, RUN_DIR / "config" / "candidate_grid.json")


## 8. Single-candidate training function

In [ ]:
def train_one_candidate(cfg, candidate_index, S_train, S_val, S_test,
                        zeros_train, zeros_val, global_seed):
    """Train one candidate on the provided paths and return metrics dict."""
    label = cfg["label"]

    keras.backend.clear_session()
    # Model seed: original convention, shifted by outer seed for independent multi-seed runs
    set_all_seeds(global_seed + 10_000 + candidate_index)

    model = make_hedger_model(cfg)
    _ = model(tf.convert_to_tensor(S_train[:2]), training=False)  # build weights
    model.compile(optimizer=optimizers.Adam(learning_rate=cfg["learning_rate"]),
                  loss=mse_hedge_loss)

    cb_list = [
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=CFG.reduce_lr_patience, min_lr=1e-5, verbose=0,
        ),
        callbacks.EarlyStopping(
            monitor="val_loss", patience=CFG.early_stopping_patience,
            restore_best_weights=True, verbose=0,
        ),
    ]

    t0 = time.time()
    hist = model.fit(
        S_train, zeros_train,
        validation_data=(S_val, zeros_val),
        epochs=CFG.max_epochs,
        batch_size=cfg["batch_size"],
        callbacks=cb_list,
        verbose=0,
    )
    runtime = time.time() - t0

    # Evaluate
    test_errors = model.predict(S_test, verbose=0)[:, 0]
    pi_nn = float(model.pi.numpy())
    best_val_loss = float(np.min(hist.history["val_loss"]))
    test_rmse = float(np.sqrt(np.mean(test_errors**2)))

    return {
        "label": label,
        "val_loss": best_val_loss,
        "test_rmse": test_rmse,
        "learned_premium": pi_nn,
        "runtime_s": runtime,
    }

## 9. Multi-seed outer loop

For each seed:
1. Generate fresh train / val / test paths.
2. Run all 12 candidates.
3. Store per-seed results.

After all seeds, compute mean ± std across seeds for each candidate.

In [ ]:
# Storage: list of DataFrames, one per seed
per_seed_results = []

for seed in SEEDS:
    append_log(f"\n{'='*60}")
    append_log(f"Starting seed {seed}")
    append_log(f"{'='*60}")

    # Fresh paths for this seed
    S_train = simulate_gbm(CFG.M_train, CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, seed=seed)
    S_val   = simulate_gbm(CFG.M_val,   CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, seed=seed + 1000)
    S_test  = simulate_gbm(CFG.M_test,  CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, seed=seed + 2000)

    zeros_train = np.zeros((CFG.M_train, 1), dtype=np.float32)
    zeros_val   = np.zeros((CFG.M_val,   1), dtype=np.float32)

    seed_rows = []
    for i, cfg in enumerate(TABLE1_CANDIDATES):
        try:
            metrics = train_one_candidate(
                cfg, i, S_train, S_val, S_test,
                zeros_train, zeros_val, global_seed=seed
            )
            metrics["seed"] = seed
            seed_rows.append(metrics)
            append_log(
                f"  [{i+1:2d}/{len(TABLE1_CANDIDATES)}] {cfg['label']}: "
                f"val_loss={metrics['val_loss']:.4e}  "
                f"test_rmse={metrics['test_rmse']:.6f}  "
                f"premium={metrics['learned_premium']:.6f}  "
                f"time={metrics['runtime_s']:.1f}s"
            )
        except Exception as e:
            append_log(f"  ERROR in {cfg['label']}: {repr(e)}")
            traceback.print_exc()

    seed_df = pd.DataFrame(seed_rows)
    seed_df.to_csv(RUN_DIR / "results" / f"seed_{seed}_results.csv", index=False)
    per_seed_results.append(seed_df)
    append_log(f"Seed {seed} complete.")

append_log("\nAll seeds complete.")

# Combined row-level output for appendix tables and runner traceability.
all_seed_rows = pd.concat(per_seed_results, ignore_index=True) if len(per_seed_results) else pd.DataFrame()
all_seed_rows.to_csv(RUN_DIR / "results" / "multiseed_per_seed_results.csv", index=False)

## 10. Build summary tables

### 10a. Individual seed tables (same columns as Table 1)

In [ ]:
for df, seed in zip(per_seed_results, SEEDS):
    print(f"\n{'='*80}")
    print(f"SEED {seed} — Individual results (Table 1 format)")
    print(f"{'='*80}")
    display_df = df[['label', 'val_loss', 'test_rmse', 'learned_premium', 'runtime_s']].copy()
    display_df.columns = ['Architecture', 'Validation loss', 'Test RMSE', 'Learned premium', 'Runtime (s)']
    display_df = display_df.sort_values('Validation loss').reset_index(drop=True)
    print(display_df.to_string(index=False, float_format=lambda x: f'{x:.6e}' if abs(x) < 0.01 else f'{x:.6f}'))

### 10b. Mean ± std summary table across all seeds

In [ ]:
all_results = pd.concat(per_seed_results, ignore_index=True)

# Compute mean and std per candidate
agg = all_results.groupby('label').agg(
    val_loss_mean=('val_loss',        'mean'),
    val_loss_std=('val_loss',         'std'),
    test_rmse_mean=('test_rmse',      'mean'),
    test_rmse_std=('test_rmse',       'std'),
    premium_mean=('learned_premium',  'mean'),
    premium_std=('learned_premium',   'std'),
    runtime_mean=('runtime_s',        'mean'),
).reset_index()

# Preserve original candidate order
order = [c['label'] for c in TABLE1_CANDIDATES]
agg['order'] = agg['label'].map({l: i for i, l in enumerate(order)})
agg = agg.sort_values('order').drop(columns='order').reset_index(drop=True)

# Format as mean ± std strings for display
summary = pd.DataFrame()
summary['Architecture'] = agg['label']
summary['Val loss (mean ± std)'] = agg.apply(
    lambda r: f"{r.val_loss_mean:.4e} ± {r.val_loss_std:.2e}", axis=1)
summary['Test RMSE (mean ± std)'] = agg.apply(
    lambda r: f"{r.test_rmse_mean:.6f} ± {r.test_rmse_std:.2e}", axis=1)
summary['Learned premium (mean ± std)'] = agg.apply(
    lambda r: f"{r.premium_mean:.6f} ± {r.premium_std:.2e}", axis=1)
summary['Mean runtime (s)'] = agg['runtime_mean'].round(1)

print(f"\n{'='*80}")
print(f"MEAN ± STD ACROSS {len(SEEDS)} SEEDS — Summary Table (Table 1 format)")
print(f"{'='*80}")
print(summary.to_string(index=False))

# Save to CSV
agg.to_csv(RUN_DIR / "results" / "multiseed_summary_numeric.csv", index=False)
summary.to_csv(RUN_DIR / "results" / "multiseed_summary_formatted.csv", index=False)

### 10c. Architecture selection: winner by mean validation loss

In [ ]:
winner_row = agg.sort_values('val_loss_mean').iloc[0]
runner_up  = agg.sort_values('val_loss_mean').iloc[1]

print(f"\n{'='*80}")
print("ARCHITECTURE SELECTION RESULT")
print(f"{'='*80}")
print(f"Winner (lowest mean val loss):  {winner_row['label']}")
print(f"  Mean val loss:  {winner_row['val_loss_mean']:.6e}  (std {winner_row['val_loss_std']:.2e})")
print(f"  Mean test RMSE: {winner_row['test_rmse_mean']:.6f}  (std {winner_row['test_rmse_std']:.2e})")
print()
print(f"Runner-up:                      {runner_up['label']}")
print(f"  Mean val loss:  {runner_up['val_loss_mean']:.6e}  (std {runner_up['val_loss_std']:.2e})")
print(f"  Mean test RMSE: {runner_up['test_rmse_mean']:.6f}  (std {runner_up['test_rmse_std']:.2e})")
print()

margin = winner_row['val_loss_mean'] - runner_up['val_loss_mean']
pooled_std = np.sqrt((winner_row['val_loss_std']**2 + runner_up['val_loss_std']**2) / 2)
print(f"Mean val loss margin (winner - runner-up): {margin:.2e}")
print(f"Pooled std: {pooled_std:.2e}")
if pooled_std > 0:
    print(f"Margin / pooled std: {margin / pooled_std:.2f}")
    if abs(margin) < pooled_std:
        print("NOTE: margin < 1 pooled std — top two are statistically indistinguishable.")
        print("Report both as jointly competitive; apply tiebreaker (lower parameter count).")

# Per-seed win count for top two
print(f"\nPer-seed winner by val loss:")
for df, seed in zip(per_seed_results, SEEDS):
    seed_winner = df.sort_values('val_loss').iloc[0]['label']
    print(f"  Seed {seed}: {seed_winner}")

# Save selection result
selection_result = {
    "seeds_used": SEEDS,
    "selection_criterion": "mean validation loss across seeds",
    "winner": winner_row['label'],
    "winner_val_loss_mean": float(winner_row['val_loss_mean']),
    "winner_val_loss_std":  float(winner_row['val_loss_std']),
    "winner_test_rmse_mean": float(winner_row['test_rmse_mean']),
    "winner_test_rmse_std":  float(winner_row['test_rmse_std']),
    "runner_up": runner_up['label'],
    "runner_up_val_loss_mean": float(runner_up['val_loss_mean']),
    "margin_val_loss": float(margin),
    "pooled_std_val_loss": float(pooled_std),
}
with open(RUN_DIR / "results" / "selection_result.json", "w") as f:
    json.dump(selection_result, f, indent=2)

append_log(f"Winner: {winner_row['label']}")
append_log(f"Runner-up: {runner_up['label']}")
append_log(f"Results saved to {RUN_DIR / 'results'}")

# Save a short appendix interpretation note.
selected_64x3 = agg.loc[agg['label'] == 'shared_norm_64u_3L_tanh_sigmoid']
selected_64x2_relu = agg.loc[agg['label'] == 'shared_norm_64u_2L_relu_sigmoid']
interpretation_lines = [
    '# Appendix interpretation note',
    '',
    'This multi-seed run is a robustness check for the architecture-selection stage.',
    'It uses the same Black--Scholes setup and batch-size protocol as the original architecture-selection experiment.',
    '',
    f"Winner by mean validation loss: `{winner_row['label']}`.",
    f"Runner-up by mean validation loss: `{runner_up['label']}`.",
    '',
    'The intended report interpretation is conservative: use this table to show that the chosen shared normalized MLP family is stable across seeds, not to claim a unique globally optimal architecture.',
]
if len(selected_64x3) == 1:
    r = selected_64x3.iloc[0]
    interpretation_lines.extend([
        '',
        'Report-selected architecture:',
        f"- `shared_norm_64u_3L_tanh_sigmoid`: mean val loss {r['val_loss_mean']:.6e}, mean test RMSE {r['test_rmse_mean']:.6f}.",
    ])
if len(selected_64x2_relu) == 1:
    r = selected_64x2_relu.iloc[0]
    interpretation_lines.extend([
        '',
        'Close ReLU competitor:',
        f"- `shared_norm_64u_2L_relu_sigmoid`: mean val loss {r['val_loss_mean']:.6e}, mean test RMSE {r['test_rmse_mean']:.6f}.",
    ])
interpretation_lines.extend([
    '',
    'Suggested wording if the top models are close:',
    '',
    '> Across three independent seeds, the shared normalized MLP architectures remain the leading family. The selected 64x3 tanh sigmoid model and the 64x2 ReLU sigmoid competitor achieve similar error levels, so the architecture choice is interpreted as selecting a reliable representative architecture rather than identifying a unique global optimum.',
])
with open(RUN_DIR / 'results' / 'appendix_interpretation_note.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(interpretation_lines) + '\n')


In [ ]:
# =============================================================================
# OPTIONAL SANITY CHECK: original train/val/test seeds 42/99/123
# This is disabled by default because the appendix run should contain only the
# pre-committed three-seed robustness experiment above. Enable manually only if
# you specifically want to check recoverability of the original one-seed ranking.
# =============================================================================

if RUN_ORIGINAL_SEED_SANITY_CHECK:
    ORIGINAL_TRAIN_SEED = 42
    ORIGINAL_VAL_SEED   = 99
    ORIGINAL_TEST_SEED  = 123
    ORIGINAL_GLOBAL_SEED = 2026

    append_log("Running optional sanity check with original seeds 42/99/123...")

    S_train_orig = simulate_gbm(CFG.M_train, CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, seed=ORIGINAL_TRAIN_SEED)
    S_val_orig   = simulate_gbm(CFG.M_val,   CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, seed=ORIGINAL_VAL_SEED)
    S_test_orig  = simulate_gbm(CFG.M_test,  CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, seed=ORIGINAL_TEST_SEED)

    zeros_train_orig = np.zeros((CFG.M_train, 1), dtype=np.float32)
    zeros_val_orig   = np.zeros((CFG.M_val,   1), dtype=np.float32)

    sanity_rows = []
    for i, cfg in enumerate(TABLE1_CANDIDATES):
        try:
            metrics = train_one_candidate(
                cfg, i, S_train_orig, S_val_orig, S_test_orig,
                zeros_train_orig, zeros_val_orig, global_seed=ORIGINAL_GLOBAL_SEED
            )
            metrics["seed"] = "original_42_99_123"
            sanity_rows.append(metrics)
        except Exception as e:
            append_log(f"  ERROR in {cfg['label']}: {repr(e)}")
            traceback.print_exc()

    sanity_df = pd.DataFrame(sanity_rows).sort_values('val_loss').reset_index(drop=True)
    sanity_df.to_csv(RUN_DIR / "results" / "sanity_check_original_seeds.csv", index=False)
    print(sanity_df[['label', 'val_loss', 'test_rmse', 'runtime_s']].to_string(index=False))
else:
    print("Optional original-seed sanity check skipped. Set RUN_ORIGINAL_SEED_SANITY_CHECK=True to run it.")


In [ ]:
# ============================================================
# Final output manifest
# ============================================================
output_rows = []
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file():
        output_rows.append({
            'relative_path': str(p.relative_to(RUN_DIR)),
            'size_bytes': p.stat().st_size,
        })
manifest = pd.DataFrame(output_rows)
manifest.to_csv(RUN_DIR / 'results' / 'output_manifest.csv', index=False)
print(f'Saved {len(output_rows)} output files under {RUN_DIR}')
print(RUN_DIR)
